一个展示 minGPT 最简单用法的精简演示。配置为在大约一分钟内即可在 Macbook Air 上平稳运行。

In [1]:
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from mingpt.utils import set_seed
set_seed(3407)

In [2]:
import pickle

class SortDataset(Dataset):
    """ 
    排序问题的数据集。例如，对于长度为 6 的问题：
    输入: 0 0 2 1 0 1 -> 输出: 0 0 0 1 1 2
    连接后输入到 Transformer 的形式如下：
    input:  0 0 2 1 0 1 0 0 0 1 1
    output: I I I I I 0 0 0 1 1 2
    其中 I 代表 “忽略 (ignore)”，因为 Transformer 正在读取输入序列
    """

    def __init__(self, split, length=6, num_digits=3):
        assert split in {'train', 'test'}
        self.split = split
        self.length = length
        self.num_digits = num_digits
    
    def __len__(self):
        return 10000 # 任意设定，我们将使用拒绝采样实时生成样本，所以这并不重要。实际上，这个数字是每个 epoch 的迭代次数，如果需要训练更久，我们可以根据需要将其设为任意大。
    
    def get_vocab_size(self):
        return self.num_digits
    
    def get_block_size(self):
        # 输入到 Transformer 的序列长度，
        # 包含连接后的输入和输出，但减去 1 是因为
        # Transformer 从最后一个输入元素开始进行预测
        return self.length * 2 - 1

    def __getitem__(self, idx):
        
        # 使用拒绝采样从所需的数据集中生成一个输入样本
        while True:
            # 生成一些随机整数
            inp = torch.randint(self.num_digits, size=(self.length,), dtype=torch.long)
            # 在一半的时间里，让我们尝试增加具有大量重复数字的样本数量，
            # 因为这似乎是模型在训练后期比较困难的地方，而且这类样本比较少见
            if torch.rand(1).item() < 0.5:
                if inp.unique().nelement() > self.length // 2:
                    # 唯一数字太多，重新采样
                    continue
            # 根据哈希值确定生成的样本属于训练集还是测试集
            h = hash(pickle.dumps(inp.tolist()))
            inp_split = 'test' if h % 4 == 0 else 'train' # 将 25% 的样本指定为测试集
            if inp_split == self.split:
                break # ok
        
        # 解决任务：即进行排序
        sol = torch.sort(inp)[0]

        # 将问题描述和解决方案连接起来
        cat = torch.cat((inp, sol), dim=0)

        # 输入到 Transformer 的将是偏移后的序列
        x = cat[:-1].clone()
        y = cat[1:].clone()
        # 我们只想在输出位置进行预测，因此屏蔽掉输入位置的损失
        y[:self.length-1] = -1
        return x, y


In [3]:
# 打印数据集中的一个示例
train_dataset = SortDataset('train')
test_dataset = SortDataset('test')
x, y = train_dataset[0]
for a, b in zip(x,y):
    print(int(a),int(b))

1 -1
0 -1
1 -1
0 -1
0 -1
0 0
0 0
0 0
0 0
0 1
1 1


In [4]:
# 创建一个 GPT 实例
from mingpt.model import GPT

model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()
model = GPT(model_config)

number of parameters: 0.09M


In [5]:
# 创建一个 Trainer 对象
from mingpt.trainer import Trainer

train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4 # 我们使用的模型非常小，所以可以训练得快一点
train_config.max_iters = 2000
train_config.num_workers = 0
trainer = Trainer(train_config, model, train_dataset)

running on device mps


In [6]:
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

trainer.run()

iter_dt 0.00ms; iter 0: train loss 1.05862
iter_dt 12.62ms; iter 100: train loss 0.18295
iter_dt 12.08ms; iter 200: train loss 0.07137
iter_dt 11.67ms; iter 300: train loss 0.04582
iter_dt 13.57ms; iter 400: train loss 0.01320
iter_dt 12.10ms; iter 500: train loss 0.06784
iter_dt 12.61ms; iter 600: train loss 0.02651
iter_dt 12.20ms; iter 700: train loss 0.01173
iter_dt 13.16ms; iter 800: train loss 0.00754
iter_dt 13.57ms; iter 900: train loss 0.03257
iter_dt 11.40ms; iter 1000: train loss 0.02473
iter_dt 11.07ms; iter 1100: train loss 0.00892
iter_dt 11.68ms; iter 1200: train loss 0.02675
iter_dt 12.63ms; iter 1300: train loss 0.01401
iter_dt 13.19ms; iter 1400: train loss 0.00350
iter_dt 12.85ms; iter 1500: train loss 0.00303
iter_dt 13.18ms; iter 1600: train loss 0.03104
iter_dt 12.43ms; iter 1700: train loss 0.00057
iter_dt 11.84ms; iter 1800: train loss 0.00168
iter_dt 10.94ms; iter 1900: train loss 0.01465


In [7]:
# 现在让我们进行一些评估
model.eval();

In [8]:
def eval_split(trainer, split, max_batches):
    dataset = {'train':train_dataset, 'test':test_dataset}[split]
    n = train_dataset.length # 粗暴的直接访问，耸肩
    results = []
    mistakes_printed_already = 0
    loader = DataLoader(dataset, batch_size=100, num_workers=0, drop_last=False)
    for b, (x, y) in enumerate(loader):
        x = x.to(trainer.device)
        y = y.to(trainer.device)
        # 单独分离出输入模式
        inp = x[:, :n]
        sol = y[:, -n:]
        # 让模型采样序列的剩余部分
        cat = model.generate(inp, n, do_sample=False) # 使用贪婪的最大概率值，不进行采样
        sol_candidate = cat[:, n:] # 分离出填充后的序列
        # 将预测序列与真实序列进行比较
        correct = (sol == sol_candidate).all(1).cpu() # 软件 1.0 与软件 2.0 在这一行展开对决，哈哈
        for i in range(x.size(0)):
            results.append(int(correct[i]))
            if not correct[i] and mistakes_printed_already < 3: # 最多只打印 3 个错误以了解情况
                mistakes_printed_already += 1
                print("GPT claims that %s sorted is %s but gt is %s" % (inp[i].tolist(), sol_candidate[i].tolist(), sol[i].tolist()))
        if max_batches is not None and b+1 >= max_batches:
            break
    rt = torch.tensor(results, dtype=torch.float)
    print("%s final score: %d/%d = %.2f%% correct" % (split, rt.sum(), len(results), 100*rt.mean()))
    return rt.sum()

# 将大量来自训练集和测试集的样本输入模型，并验证输出的正确性
with torch.no_grad():
    train_score = eval_split(trainer, 'train', max_batches=50)
    test_score  = eval_split(trainer, 'test',  max_batches=50)

train final score: 5000/5000 = 100.00% correct
test final score: 5000/5000 = 100.00% correct


In [9]:
# 让我们也将一个给定的随机序列输入模型进行测试
n = train_dataset.length # 粗暴的直接访问，耸肩
inp = torch.tensor([[0, 0, 2, 1, 0, 1]], dtype=torch.long).to(trainer.device)
assert inp[0].nelement() == n
with torch.no_grad():
    cat = model.generate(inp, n, do_sample=False)
sol = torch.sort(inp[0])[0]
sol_candidate = cat[:, n:]
print('input sequence  :', inp.tolist())
print('predicted sorted:', sol_candidate.tolist())
print('gt sort         :', sol.tolist())
print('matches         :', bool((sol == sol_candidate).all()))

input sequence  : [[0, 0, 2, 1, 0, 1]]
predicted sorted: [[0, 0, 0, 1, 1, 2]]
gt sort         : [0, 0, 0, 1, 1, 2]
matches         : True
